In [ ]:
from typing import Callable

import numpy as np
from torch.utils.data import BatchSampler, DataLoader, Dataset

In [ ]:
class DummyPatchSpecs:
    data_index: int

In [ ]:
class FifoImageStackLoader:

    def __init__(self, max_files_loaded: int = 1):
        self.loaded_image_stacks: list[LazyImageStack] = []
        self.max_files_loaded = max_files_loaded

    @property
    def currently_loaded(self):
        return len(self.loaded_image_stacks)

    def free(self):
        if self.currently_loaded >= self.max_files_loaded:
            image_stack_to_close = self.loaded_image_stacks[0]  # FIFO
            image_stack_to_close.close()

    def notify_load(self, image_stack: "LazyImageStack"):
        self.free()
        self.loaded_image_stacks.append(image_stack)

    def notify_close(self, image_stack: "LazyImageStack"):
        loaded_paths = [img_stack.file_id for img_stack in self.loaded_image_stacks]
        index = loaded_paths.index(image_stack.file_id) #TODO: this will be file path
        self.loaded_image_stacks.pop(index)

    def close_all(self):
        loaded_image_stacks = self.loaded_image_stacks.copy()
        for image_stack in loaded_image_stacks:
            image_stack.close()


class LazyImageStack:

    def __init__(
        self,
        file_id: int,
        on_load: Callable[["LazyImageStack"], None] | None = None,
        on_close: Callable[["LazyImageStack"], None] | None = None,
    ):
        self.file_id = file_id
        self.is_loaded: bool = False
        # Callback system
        self._on_load = on_load
        self._on_close = on_close

    def register_manager(self, manager: FifoImageStackLoader):
        self._on_load = manager.notify_load
        self._on_close = manager.notify_close

    def extract_patch(self):
        if not self.is_loaded:
            self.load()
        patch = f"Patch from image stack {self.file_id}"
        return patch

    def load(self):
        if self._on_load is not None:
            self._on_load(self)
        self.is_loaded = True
        print(f"--- Loaded file: {self.file_id} ---")

    def close(self):
        # TODO: raise error if not loaded?
        if self._on_close is not None:
            self._on_close(self)
        self.is_loaded = False
        print(f"--- Closed file: {self.file_id} ---")

In [ ]:
class DummyPatchingStrategy:

    def __init__(self, image_stacks: list[LazyImageStack]):
        self.patches_per_image_stack = [
            np.random.randint(8, 16) for _ in image_stacks
        ]
        self.patch_bins = np.concatenate(
            [[0], np.cumsum(self.patches_per_image_stack)]
        )

    def __getitem__(self, index: int) -> DummyPatchSpecs:
        # minus 1, because digitize returns 1 for first bin
        data_index = np.digitize(index, self.patch_bins) - 1
        return {"data_index": data_index}

In [ ]:
class DummyDataset(Dataset):

    def __init__(self, image_stacks: list[LazyImageStack]):
        self.image_stacks = image_stacks
        self.patching_strategy = DummyPatchingStrategy(image_stacks)

    def __len__(self):
        return sum(self.patching_strategy.patches_per_image_stack)

    def __getitem__(self, index):
        patch_specs = self.patching_strategy[index]
        data_index = patch_specs["data_index"]
        patch = self.image_stacks[data_index].extract_patch()
        return patch

In [ ]:
class FifoBatchSampler(BatchSampler):

    def __init__(
        self,
        dataset: DummyDataset,
        batch_size: int,
        n_files_loaded: int = 1,
        seed=int | None,
    ):
        self.dataset = dataset
        self.batch_size = batch_size
        self.n_files_loaded = n_files_loaded
        self.manager = FifoImageStackLoader(n_files_loaded)
        for image_stack in self.dataset.image_stacks:
            image_stack.register_manager(self.manager)
        self.rng = np.random.default_rng() # TODO: add seed

    # TODO: separate code into more manageable functions
    def __iter__(self):

        n_files = len(self.dataset.image_stacks)
        file_indices = np.arange(n_files)
        self.rng.shuffle(file_indices)
        file_groups = [
            file_indices[i : i + self.n_files_loaded]
            for i in range(0, n_files, self.n_files_loaded)
        ]
        remaining_indices = None
        for file_group in file_groups:
            patch_indices = np.zeros((0,))
            for file_index in file_group:
                file_patch_indices = np.arange(
                    self.dataset.patching_strategy.patch_bins[file_index],
                    self.dataset.patching_strategy.patch_bins[file_index + 1],
                )
                patch_indices = np.concatenate([patch_indices, file_patch_indices])
            self.rng.shuffle(patch_indices)

            if remaining_indices is not None:
                patch_indices = np.concatenate([remaining_indices, patch_indices])
            remaining_indices = None

            for i in range(0, len(patch_indices), self.batch_size):
                batch_indices = list(patch_indices[i : i + self.batch_size])
                if len(batch_indices) == self.batch_size:
                    yield batch_indices
                else:
                    remaining_indices = batch_indices

        if remaining_indices is not None:
            yield remaining_indices

        self.manager.close_all()

In [ ]:
image_stacks = image_stacks = [LazyImageStack(i) for i in range(10)]
dataset = DummyDataset(image_stacks)

In [ ]:
batch_sampler = FifoBatchSampler(dataset=dataset, batch_size=3, n_files_loaded=4)
dataloader = DataLoader(
    dataset=dataset,
    batch_sampler=batch_sampler,
)

In [ ]:
track_files_open = []
track_batch_size = []
files_open = sum(image_stack.is_loaded for image_stack in dataset.image_stacks)
print("--- files open: ", files_open)
for patch_batch in dataloader:
    files_open = sum(image_stack.is_loaded for image_stack in dataset.image_stacks)
    track_files_open.append(files_open)
    track_batch_size.append(len(patch_batch))
    print(patch_batch)
    print("Batch size: ", len(patch_batch), "| files open: ", files_open)
print(track_files_open)
print(track_batch_size)

In [ ]:
for i, image_stack in enumerate(dataset.image_stacks):
    print(i, image_stack.is_loaded)

In [ ]:
files_open = sum(image_stack.is_loaded for image_stack in dataset.image_stacks)
print(files_open)